[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/14_tag_2_3_tensorflow_grundlagen_cpu_gpu.ipynb)

# Tag 2/3 - 14 TensorFlow Grundlagen: Rechnen, Graphen, CPU und GPU

TensorFlow ist mehr als `model.fit(...)`. Unter der Keras-Oberfläche ist TensorFlow eine Bibliothek für numerisches Rechnen:

- Tensoren erstellen und transformieren,
- Matrixmultiplikationen ausführen,
- automatische Ableitungen berechnen,
- Rechenschritte als Graph optimieren,
- Operationen auf CPU, GPU oder TPU ausführen.

In Colab kann man unter **Runtime > Change runtime type > T4 GPU** eine GPU aktivieren und die Unterschiede in diesem Notebook ausprobieren.


## Lesepfad für dieses Notebook

Dieses Notebook ist kein Modelltraining, sondern zeigt TensorFlow als Rechenbibliothek. Die Beispiele bauen aufeinander auf:

1. Tensoren sind die Datenform.
2. Operationen wie Matrixmultiplikation und ReLU rechnen auf ganzen Tensoren.
3. `GradientTape` berechnet Ableitungen.
4. `tf.function` kann Python-Funktionen als TensorFlow-Graph optimieren.
5. CPU/GPU wird am Ende mit derselben Matrixoperation verglichen.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

print("TensorFlow:", tf.__version__)
print("Physische Geräte:")
for device in tf.config.list_physical_devices():
    print(" -", device)


## Tensoren

Ein Tensor ist eine mehrdimensionale Zahlentabelle:

- Skalar: keine Achse, zum Beispiel `3.0`
- Vektor: eine Achse, zum Beispiel `[1, 2, 3]`
- Matrix: zwei Achsen
- Batch von Bildern: häufig vier Achsen `(batch, height, width, channels)`

TensorFlow-Operationen arbeiten meist auf ganzen Tensoren statt auf einzelnen Python-Zahlen.


In [ ]:
# x ist ein Mini-Batch mit zwei Samples und zwei Features.
x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
# w und b sind wie Gewichte und Bias in einem Dense-Layer.
w = tf.constant([[2.0], [-1.0]])
b = tf.constant([0.5])

# @ ist Matrixmultiplikation. Das ist der Kern vieler neuronaler Netze.
linear = x @ w + b
activation = tf.nn.relu(linear)

print("x shape:", x.shape)
print("linear:")
print(linear.numpy())
print("relu(linear):")
print(activation.numpy())


## Broadcasting und Vektorisierung

In Deep Learning schreiben wir selten Schleifen über einzelne Samples. Stattdessen rechnen wir mit ganzen Batches.

Broadcasting bedeutet: TensorFlow erweitert kleinere Tensoren logisch auf passende Form, ohne dass wir sie wirklich kopieren müssen.


In [ ]:
batch = tf.random.normal(shape=(5, 3), seed=RANDOM_STATE)
feature_mean = tf.reduce_mean(batch, axis=0)
centered = batch - feature_mean

pd.DataFrame(
    {
        "feature_mean": feature_mean.numpy(),
        "centered_mean_after_subtraction": tf.reduce_mean(centered, axis=0).numpy(),
    }
)


## Automatische Ableitung mit GradientTape

TensorFlow kann Ableitungen automatisch berechnen. Das ist der Kern von Backpropagation.

Im Beispiel minimieren wir eine einfache Funktion: `loss = (w - 3)^2`. Die Ableitung sagt uns, in welche Richtung `w` verändert werden muss.


In [ ]:
w = tf.Variable(10.0)

rows = []
for step in range(8):
    with tf.GradientTape() as tape:
        loss = (w - 3.0) ** 2
    grad = tape.gradient(loss, w)
    w.assign_sub(0.2 * grad)
    rows.append({"step": step, "w": float(w.numpy()), "loss": float(loss.numpy()), "gradient": float(grad.numpy())})

pd.DataFrame(rows)


## `tf.function`: Python-Code als TensorFlow-Graph

Normale Python-Funktionen laufen Schritt für Schritt im Python-Interpreter. Mit `@tf.function` versucht TensorFlow, daraus einen optimierten Graph zu bauen.

Das kann schneller sein, macht Debugging aber manchmal schwerer. Für den Unterricht ist wichtig: Keras nutzt intern viele dieser Graph-Mechanismen.


In [ ]:
def eager_matmul(a, b):
    return tf.nn.relu(a @ b)


@tf.function
def graph_matmul(a, b):
    return tf.nn.relu(a @ b)


a = tf.random.normal((512, 512), seed=RANDOM_STATE)
b = tf.random.normal((512, 512), seed=RANDOM_STATE + 1)

# Erster Aufruf baut den Graphen.
_ = graph_matmul(a, b)

print("Eager result shape:", eager_matmul(a, b).shape)
print("Graph result shape:", graph_matmul(a, b).shape)


## CPU/GPU-Vergleich

GPUs sind besonders stark bei großen Matrixoperationen. Genau diese Operationen passieren in neuronalen Netzen ständig.

Der folgende Benchmark ist absichtlich einfach: mehrere Matrixmultiplikationen auf CPU und, falls vorhanden, GPU. In Colab mit T4 sollte man meistens einen Unterschied sehen. Auf kleinen Matrizen kann die CPU aber gleich schnell oder schneller sein, weil GPU-Startkosten und Datenbewegung zählen.


In [ ]:
def benchmark_matmul(device_name, size=2048, repeats=10):
    # Diese Funktion wird unten einmal für CPU und, falls vorhanden, einmal für GPU aufgerufen.
    # device_name ist z. B. "/CPU:0" oder "/GPU:0".
    with tf.device(device_name):
        a = tf.random.normal((size, size), seed=RANDOM_STATE)
        b = tf.random.normal((size, size), seed=RANDOM_STATE + 1)

        # Warmup: Der erste Aufruf kann Initialisierungsaufwand enthalten.
        warmup = a @ b
        warmup.numpy()

        start = time.perf_counter()
        for _ in range(repeats):
            c = a @ b
            # .numpy() zwingt TensorFlow, das Ergebnis wirklich fertig zu berechnen.
            c.numpy()
        duration = time.perf_counter() - start
    return duration / repeats


devices_to_test = ["/CPU:0"]
if tf.config.list_physical_devices("GPU"):
    devices_to_test.append("/GPU:0")

benchmark_rows = []
for device_name in devices_to_test:
    seconds = benchmark_matmul(device_name)
    benchmark_rows.append({"device": device_name, "seconds_per_matmul": seconds})

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df


In [ ]:
ax = benchmark_df.plot.bar(x="device", y="seconds_per_matmul", legend=False, figsize=(6, 4))
ax.set_ylabel("Sekunden pro Matrixmultiplikation")
ax.set_title("Einfacher CPU/GPU-Vergleich")
plt.xticks(rotation=0)


## Limitierungen

- Benchmarks in Notebooks sind nur grobe Orientierung. Andere Nutzer auf derselben Colab-Maschine, Aufwärmphase und Hintergrundprozesse beeinflussen die Zahlen.
- Der erste Aufruf einer `tf.function` enthält Tracing-Aufwand. Deshalb sollte man den ersten Aufruf nicht als reine Rechenzeit interpretieren.
- Kleine Operationen profitieren oft kaum von der GPU. Der Overhead kann größer sein als der Rechenvorteil.
- GPU ist nicht automatisch schneller, wenn Daten ständig zwischen CPU und GPU kopiert werden.
- TensorFlow kann nicht jede Python-Logik effizient in einen Graph übersetzen. Tensor-Operationen sind der zentrale Pfad.
